<div dir="rtl">

# 🎯 06 - VectorStore Retrievers & Advanced Search Types

## ما هو الـ Retriever في LangChain؟
- **Retriever (المُسترجِع)** هو واجهة برمجية موحدة في LangChain تقبل نص استعلام غير منظم (`str`) وتُرجع قائمة من المستندات ذات الصلة (`list[Document]`).
- يُعد الـ Retriever حجر الأساس في سلاسل الـ **LCEL (LangChain Expression Language)** والـ **RAG Pipelines**.
- يمكن تحويل **أي VectorStore** إلى Retriever بلمسة واحدة عبر دالة `.as_retriever()`.

---

### 🔍 استراتيجيات البحث المتاحة (Search Types):
1. **`similarity` (التشابه القياسي)**: استرجاع أعلى K مستندات مطابقة لأقرب مسافة متجهات.
2. **`similarity_score_threshold` (حد أدنى لدرجة الثقة)**: استبعاد أي مستندات تقل درجة تطابقها عن نسبة معينة.
3. **`mmr` (Maximal Marginal Relevance - التنوع وأقصى صلة)**:
   - يحل مشكلة **التكرار وتشابه المقاطع المسترجعة**.
   - يختار المستندات التي تحقق توازناً بين **الصلة بالاستعلام** و**التنوع والتباين فيما بينها**.

</div>


<div dir="rtl">

### 1️⃣ تجهيز البيئة ونموذج التضمين

</div>


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

load_dotenv(find_dotenv())

# تهيئة نموذج التضمين
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("✅ تم تحميل نموذج التضمين بنجاح!")


<div dir="rtl">

### 2️⃣ بناء قاعدة مستندات لاختبار التكرار والتنوع (MMR)
سننشئ مستندات متعددة تتحدث عن نفس الموضوع بصيغ متطابقة ومستندات أخرى تقدم تفاصيل إضافية.

</div>


In [ ]:
documents = [
    Document(page_content="الـ RAG يدمج بين استرجاع المستندات وتوليد النصوص لتقليل هلوسة النماذج اللغوية.", metadata={"doc_id": 1}),
    Document(page_content="تقنية RAG تجمع نماذج التوليد مع محركات استرجاع البيانات لتفادي الهلوسة.", metadata={"doc_id": 2}),
    Document(page_content="نظام RAG يسترجع مقاطع سياقية من قاعدة المتجهات ويرسلها كمدخل إضافي للنموذج.", metadata={"doc_id": 3}),
    Document(page_content="تقنيات التقييم في RAG تشمل RAGAS و TruLens لقياس الدقة والارتباط الدلالي للسياق.", metadata={"doc_id": 4}),
    Document(page_content="تعد مرحلة الـ Chunking وتقسيم المستندات خطوة حاسمة لنجاح منظومة الـ RAG.", metadata={"doc_id": 5}),
    Document(page_content="تعلم قيادة الدراجات الهوائية يحتاج إلى توازن وتركيز مستمر في الطرقات.", metadata={"doc_id": 6})
]

# بناء مستودع FAISS
vector_store = FAISS.from_documents(documents, embeddings)
print("✅ تم تجهيز المستودع المتجهي!")


<div dir="rtl">

### 3️⃣ النوع الأول: البحث بالتشابه القياسي (`search_type="similarity"`)
يجلب أعلى 3 نتائج مطابقة. لاحظ كيف قد تكون النتائج مكررة في المعنى.

</div>


In [ ]:
similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

query = "ما هي فوائد تقنية RAG في الذكاء الاصطناعي؟"
results_sim = similarity_retriever.invoke(query)

print("🔎 نتائج التشابه القياسي (قد تحتوي على نصوص مكررة المعنى):")
for i, doc in enumerate(results_sim, 1):
    print(f"{i}. [Doc ID {doc.metadata['doc_id']}]: {doc.page_content}")


<div dir="rtl">

### 4️⃣ النوع الثاني: البحث مع التنوع الأقصى (`search_type="mmr"`)
يستخدم خوارزمية **Maximal Marginal Relevance**:
- `fetch_k`: عدد المستندات الأولية التي يتم جلبها للفحص (مثلاً 6).
- `k`: عدد المستندات النهائية المختارة (مثلاً 3).
- `lambda_mult`: معامل التوازن بين الصلة (1.0) والتنوع الكامل (0.0). القيمة 0.5 تعطي توازناً مثالياً.

</div>


In [ ]:
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 6,
        "lambda_mult": 0.5
    }
)

results_mmr = mmr_retriever.invoke(query)

print("🌟 نتائج الـ MMR (تنوع في الأفكار وتقليل التكرار):")
for i, doc in enumerate(results_mmr, 1):
    print(f"{i}. [Doc ID {doc.metadata['doc_id']}]: {doc.page_content}")


<div dir="rtl">

### 5️⃣ النوع الثالث: الفلترة بنسبة الثقة الأدنى (`search_type="similarity_score_threshold"`)
استبعاد أي نتائج غير ذات صلة تتجاوز عتبة معينة.

</div>


In [ ]:
threshold_retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.3, "k": 3}
)

results_thresh = threshold_retriever.invoke("كيف أصنع البيتزا الإيطالية؟")

print(f"📊 عدد النتائج المسترجعة لاستعلام غير موجود: {len(results_thresh)}")
if len(results_thresh) == 0:
    print("✅ تم استبعاد جميع النتائج بنجاح لأنها لا تحقق الحد الأدنى من الصلة!")


<div dir="rtl">

### 6️⃣ استخدام الـ Retriever داخل سلسلة LCEL (RAG Chain)
دمج الـ Retriever بسلاسة مع Prompt وتنسيق المستندات في بايبلاين موحد.

</div>


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

# دالة لتنسيق المستندات المسترجعة كنص واحد
def format_docs(docs):
    return "\n\n".join([f"- {doc.page_content}" for doc in docs])

# قالب الـ Prompt للـ RAG
template = """أنت مساعد ذكي. أجب على السؤال التالي بالاعتماد فقط على السياق المرفق:

السياق:
{context}

السؤال: {question}

الإجابة المختصرة:"""

prompt = PromptTemplate.from_template(template)

# بناء الـ RAG Chain عبر LCEL
rag_chain = (
    {"context": mmr_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
)

# معاينة المدخلات المهيأة بعد استرجاع السياق
prepared_prompt = rag_chain.invoke("ما هو دور الـ RAG وكيف يتم تقييمه؟")
print("📋 الـ Prompt النهائي مع السياق المسترجع:")
print(prepared_prompt.text)
